# Push-T variable goal, run 4: BC teacher distilled from Jimmy's PPO teacher

The group uses **one** variable-goal environment for all transformer data: Jimmy's `PushTVarGoal-v1`
(`state_rl/pusht_vargoal.py`, branch `jimmy-state-rl`, commit `c87d8ae`). The plan's teacher pair is
**PPO (Jimmy) → distilled BC (this notebook)**. Run 3b's own environment and teachers are kept as a record only. See `state_bc/README.md`.

1. **Environment:** goal T position random per episode around (−0.156, 0): ±10 cm in x, ±20 cm in y, with a 2 cm / 3 cm band
   around each axis excluded. Quadrants by the sign of the offset: Q1 (+x,+y), Q2 (−x,+y), Q3 (−x,−y), **Q4 (+x,−y)**, the transformer's
   held-out quadrant. The movable T starts anywhere in a fixed box, 8–30 cm from the goal; the stick tip starts from a library of 600 arm poses.
2. **Expert:** Jimmy's frozen `ppo_vargoal_seed1_50M.pt`, embedded below, checked by SHA-256. It is re-evaluated here first, with his
   per-quadrant protocol, as a reproduction check and a gate.
3. **BC teacher distilled from it:** noisy expert rollouts plus one training run, exactly as run 3b. **No PPO training, no DAgger.**
4. **Per-quadrant validation and test** of both teachers with Jimmy's protocol (100 episodes per quadrant; quadrant q uses seed base + 1000·q;
   validation base 1001, test base 5001), so BC and PPO are scored on the same episodes. Freeze bar: success-once ≥ 80% in every quadrant.

| Setting | Value |
|---|---|
| Observation / controller | `obs_mode="state"` (31 numbers, goal at 21:24), `pd_joint_delta_pos` (7) |
| BC teacher | MLP 2 × 256 ReLU, tanh output, MSE, Adam 3e-4 cosine, batch 1024, 200k iterations |
| Expert data | 16,384 episodes, action noise 0 / 0.1 / 0.2 / 0.3 cycled per batch, labels always the expert's clean action |
| Simulator | ManiSkill v3.0.1 (commit `a4a4f92`), GPU physics |
| Seeds | expert data 100,000+; BC validation data 300,000+; closed-loop validation during training 1001 (all quadrants mixed); per-quadrant validation 2001–5001; test 6001–9001 (once, at the end) |

Select **Runtime → Change runtime type → L4 GPU** and run the cells in order. Upload only this notebook.
Outputs go to `MyDrive/182_final_project/state_bc/<session>/`. Estimated total: about 30–40 minutes (install about 8).

In [ ]:
from pathlib import Path
import os, sys, json, re, subprocess, datetime, shlex, shutil, base64, hashlib

MANISKILL_COMMIT = "a4a4f9272ad64b1564035874b605ceb687b63ed8"  # v3.0.1, same as the other notebooks
MS_ROOT = Path("/content/ManiSkill")
WORK_ROOT = Path("/content/pusht_state_bc")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
USE_GOOGLE_DRIVE = True
DRIVE_SUBDIR = "182_final_project/state_bc"
RUN_LABEL = "run4_bc_from_jimmy"
ENV_ID = "PushTVarGoal-v1"
JIMMY_COMMIT = "c87d8ae3669c10894d765df2d0dd4b61c010e272"

# Per-quadrant protocol shared with state_rl/pusht_state_ppo.py
QUAD_EPISODES = 100          # per quadrant
QUAD_VAL_SEED = 1001         # quadrant q uses 1001 + 1000 * q
QUAD_TEST_SEED = 5001        # quadrant q uses 5001 + 1000 * q; used once, at the end
FREEZE_BAR = 0.80            # success-once in every quadrant

# BC teacher
TEACHER_NAME = "bc_from_jimmy_ppo_seed0"
SEED = 0
ITERS = 200_000
BATCH_SIZE = 1024
LR = 3e-4
EVAL_EVERY = 25_000
EVAL_EPISODES = 200          # closed-loop validation during training, all quadrants mixed
VAL_SEED = 1001

# Expert rollouts for BC
NUM_ENVS = 1024
EXPERT_EPISODES = 16_384                # 16 batches of 1,024
EXPERT_NOISE = [0.0, 0.1, 0.2, 0.3]     # action-noise std per batch, cycled; labels stay clean
EXPERT_SEED_BASE = 100_000
VAL_EPISODES = 1024                     # noise-free expert rollouts for validation action error
VAL_SEED_BASE = 300_000

SESSION = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + RUN_LABEL
subprocess.run(["nvidia-smi"], check=True)
print("Notebook Python:", sys.version)

## 1. Save outputs to Drive

Each session gets a new folder, so reruns never overwrite earlier results.

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUTPUT_ROOT = Path("/content/drive/MyDrive") / DRIVE_SUBDIR / SESSION
else:
    OUTPUT_ROOT = WORK_ROOT / "outputs" / SESSION
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
LOG = OUTPUT_ROOT / "notebook.log"
print("Output directory:", OUTPUT_ROOT)


def run(command, show=None, **kwargs):
    """Run a command and append all output to notebook.log. Prints every line, or only lines
    matching the regex `show`. On failure, raises with the last 40 lines."""
    command = list(map(str, command))
    print("$", shlex.join(command), flush=True)
    pattern = re.compile(show) if show else None
    tail = []
    with LOG.open("a") as log:
        log.write("\n$ " + shlex.join(command) + "\n")
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            try:
                for line in process.stdout:
                    if pattern is None or pattern.search(line):
                        print(line, end="", flush=True)
                    log.write(line)
                    tail = (tail + [line.rstrip()])[-40:]
                code = process.wait()
            except BaseException:
                process.kill()
                raise
    if code:
        raise RuntimeError(f"Command failed (exit {code}). Log: {LOG}\n" + "\n".join(tail))

## 2. Install ManiSkill v3.0.1 under Python 3.11

Same pinned install as runs 1–3b and Jimmy's notebooks.

In [ ]:
run(["apt-get", "update", "-qq"])
run(["apt-get", "install", "-y", "-qq", "libvulkan1", "libvulkan-dev", "vulkan-tools", "libglvnd-dev"])
if not (MS_ROOT / ".git").exists():
    run(["git", "clone", "https://github.com/mani-skill/ManiSkill.git", MS_ROOT])
run(["git", "-C", MS_ROOT, "checkout", "--detach", MANISKILL_COMMIT])
assert subprocess.check_output(["git", "-C", str(MS_ROOT), "rev-parse", "HEAD"], text=True).strip() == MANISKILL_COMMIT

run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.11"])
TRAIN_ENV = WORK_ROOT / "python311"
TRAIN_PY = TRAIN_ENV / "bin/python"
if not TRAIN_PY.exists():
    run([sys.executable, "-m", "uv", "venv", "--python", "3.11", "--seed", TRAIN_ENV])
constraints = WORK_ROOT / "constraints.txt"
constraints.write_text("numpy==1.26.4\nopencv-python==4.11.0.86\nsapien==3.0.3\ntorch==2.6.0\n")
run([TRAIN_PY, "-m", "pip", "install", "-q", "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124"])
run([TRAIN_PY, "-m", "pip", "install", "-q", "--constraint", constraints, "-e", MS_ROOT,
     "numpy", "sapien", "h5py", "tyro", "imageio[ffmpeg]", "matplotlib"])

# Vulkan/EGL entries Colab is missing; needed for rendering evaluation videos.
for filename, library, vulkan in [
    ("/usr/share/vulkan/icd.d/nvidia_icd.json", "libGLX_nvidia.so.0", True),
    ("/usr/share/glvnd/egl_vendor.d/10_nvidia.json", "libEGL_nvidia.so.0", False),
]:
    path = Path(filename)
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        icd = {"library_path": library}
        if vulkan:
            icd["api_version"] = "1.2.155"
        path.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": icd}))
os.environ["VK_ICD_FILENAMES"] = "/usr/share/vulkan/icd.d/nvidia_icd.json"
os.environ["MS_SKIP_ASSET_DOWNLOAD_PROMPT"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
run([TRAIN_PY, "-c", "import torch, sapien, mani_skill, numpy; print('torch', torch.__version__, 'CUDA', torch.version.cuda); "
     "print('mani_skill', mani_skill.__version__); assert torch.cuda.is_available(); print(torch.cuda.get_device_name(0))"])
(OUTPUT_ROOT / "pip-freeze.txt").write_text(subprocess.check_output([str(TRAIN_PY), "-m", "pip", "freeze"], text=True))
print("Training Python:", TRAIN_PY)

## 3. Write the modules and Jimmy's files

`pusht_state_bc.py` is this repo's BC module. `pusht_vargoal.py` (the shared environment) and `pusht_state_ppo.py`
(Jimmy's preflight checks) are copied unchanged from Jimmy's branch at commit `c87d8ae`. The arm-pose library and the frozen
PPO teacher follow as base64; their SHA-256 must match the committed files.

In [ ]:
%%writefile /content/pusht_state_bc/pusht_state_bc.py
"""Full-state behavioral cloning (BC) teachers for ManiSkill Push-T.

The default environment is PushT-VariableGoal-v1 (variable_goal_env.py): the goal T's position
changes every episode, and policies see it in the state. --env-id PushTVarGoal-v1 selects the
group's shared variable-goal environment (Jimmy's state_rl/pusht_vargoal.py, which must sit next to
this file); --env-id PushT-v1 the original fixed-goal task. --goal-quadrants limits which goal
quadrants are sampled.

Subcommands (run with the ManiSkill training interpreter):

    ppo      Run ManiSkill's state PPO baseline (examples/baselines/ppo/ppo.py) with the
             variable-goal environment registered. Extra arguments go to ppo.py.
    audit    Inspect the state observation and action spaces; write observation_contract.json.
    check-env  Verify the success test follows each episode's goal (variable-goal env).
    prepare  Convert a replayed state-mode demo file (.h5 + .json) into train/validation
             transition files, split by whole episode.
    collect  Roll out a driver policy (the expert, or a BC checkpoint for DAgger) and label
             every visited state with the expert's deterministic action.
    train    Train one MLP BC teacher on one or more transition files, evaluating closed-loop
             success on validation seeds.
    eval     Evaluate a BC checkpoint, or a PPO checkpoint (--ppo), on a seed range, with
             success broken down by goal quadrant.
    eval-quadrants  Evaluate checkpoints separately in each goal quadrant with the shared
             protocol of state_rl/pusht_state_ppo.py (quadrant q uses seed + 1000 * q, no
             reconfiguration), and check the freeze bar in every quadrant.

Transition files are .npz with obs [N, d_s], actions [N, d_a] and episode_ids [N],
plus a .json sidecar describing where they came from.

Pure data/model code is importable without ManiSkill, so it can be unit tested offline.
ManiSkill and gymnasium are imported lazily inside the simulator functions.
"""

from __future__ import annotations

import argparse
import json
import math
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn

VARIABLE_GOAL_ENV_ID = "PushT-VariableGoal-v1"
SHARED_ENV_ID = "PushTVarGoal-v1"       # Jimmy's shared env, module pusht_vargoal.py (state_rl/)
GOAL_ENV_IDS = (VARIABLE_GOAL_ENV_ID, SHARED_ENV_ID)
ENV_ID = VARIABLE_GOAL_ENV_ID  # set by configure_env() from --env-id
ENV_KWARGS = {}                # set by configure_env() from --goal-quadrants / --start-quadrants
# Per-episode records exposed by the variable-goal envs and saved with collected data.
# PushTVarGoal-v1 exposes only goal_quadrant; goal_xy is then read from the goal T's pose.
EPISODE_RECORDS = ("goal_quadrant", "goal_xy", "tip_start_quadrant", "tip_start_xy", "tee_start_quadrant",
                   "tip_ik_fallback")
OBS_MODE = "state"
CONTROL_MODE = "pd_joint_delta_pos"
SIM_BACKEND = "physx_cuda"
# Features whose training std is below this are only centered, not scaled. Near-constant
# fields (for example the fixed goal position) would otherwise turn simulator noise
# into very large normalized values.
MIN_SCALED_STD = 1e-3


# --------------------------------------------------------------------------------------
# Transition files
# --------------------------------------------------------------------------------------

def save_transitions(path, obs, actions, episode_ids, meta=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, obs=np.asarray(obs, np.float32), actions=np.asarray(actions, np.float32),
                        episode_ids=np.asarray(episode_ids, np.int64))
    meta = dict(meta or {}, transitions=int(len(obs)), episodes=int(len(np.unique(episode_ids))))
    path.with_suffix(".json").write_text(json.dumps(meta, indent=2))
    return meta


def load_transitions(paths):
    """Concatenate transition files; episode ids are made unique across files."""
    obs, actions, episode_ids, offset = [], [], [], 0
    for p in paths:
        d = np.load(p)
        obs.append(d["obs"])
        actions.append(d["actions"])
        ids = d["episode_ids"].astype(np.int64)
        episode_ids.append(ids - ids.min() + offset if len(ids) else ids)
        offset = (episode_ids[-1].max() + 1) if len(ids) else offset
    return np.concatenate(obs), np.concatenate(actions), np.concatenate(episode_ids)


# --------------------------------------------------------------------------------------
# Released demos
# --------------------------------------------------------------------------------------

def load_replayed_demos(h5_path: str | Path):
    """Return a list of episodes, each a dict with obs [T, d_s], actions [T, d_a], and metadata.

    The replayed file stores T+1 observations per episode; the last one has no action
    and is dropped, as in ManiSkill's own BC baseline.
    """
    import h5py

    h5_path = Path(h5_path)
    meta = json.loads(h5_path.with_suffix(".json").read_text())
    env_kwargs = meta["env_info"]["env_kwargs"]
    if env_kwargs.get("control_mode") not in (None, CONTROL_MODE):
        raise ValueError(f"Demo control mode {env_kwargs.get('control_mode')} != {CONTROL_MODE}")
    if env_kwargs.get("obs_mode") != OBS_MODE:
        raise ValueError(f"Demo obs mode {env_kwargs.get('obs_mode')} != {OBS_MODE}; replay with -o state")
    episodes = []
    with h5py.File(h5_path, "r") as f:
        for ep in meta["episodes"]:
            group = f[f"traj_{ep['episode_id']}"]
            obs = np.asarray(group["obs"], dtype=np.float32)
            actions = np.asarray(group["actions"], dtype=np.float32)
            if obs.ndim != 2 or actions.ndim != 2 or obs.shape[0] != actions.shape[0] + 1:
                raise ValueError(f"Unexpected shapes in traj_{ep['episode_id']}: obs {obs.shape}, actions {actions.shape}")
            success = np.asarray(group["success"]) if "success" in group else None
            episodes.append(dict(
                episode_id=int(ep["episode_id"]),
                episode_seed=ep.get("episode_seed"),
                obs=obs[:-1],
                # The simulator clips commands to the action space, so clipped labels are
                # exactly what the expert executed.
                actions=np.clip(actions, -1.0, 1.0),
                raw_actions=actions,
                success_at_end=bool(success[-1]) if success is not None else bool(ep.get("success", False)),
            ))
    return episodes, meta


def split_episodes(num_episodes: int, val_fraction: float, seed: int):
    """Split episode indices into train/validation sets, whole episodes only."""
    if not 0 < val_fraction < 1:
        raise ValueError("val_fraction must be in (0, 1)")
    rng = np.random.default_rng(seed)
    order = rng.permutation(num_episodes)
    num_val = max(1, int(round(num_episodes * val_fraction)))
    return np.sort(order[num_val:]), np.sort(order[:num_val])


def compute_state_stats(states: np.ndarray):
    mean = states.mean(axis=0)
    std = states.std(axis=0)
    scale = np.where(std < MIN_SCALED_STD, 1.0, std)
    return mean.astype(np.float32), std.astype(np.float32), scale.astype(np.float32)


def prepare(args):
    episodes, meta = load_replayed_demos(args.h5)
    if args.max_episodes:
        episodes = episodes[:args.max_episodes]
    train_idx, val_idx = split_episodes(len(episodes), args.val_fraction, args.split_seed)
    out = Path(args.out_dir)
    source = dict(source="released PPO demos, replayed to state obs", source_h5=str(args.h5),
                  source_env_info=meta["env_info"], source_commit=meta.get("commit_info"),
                  split_seed=args.split_seed, val_fraction=args.val_fraction)
    for name, idx in (("demos_train", train_idx), ("demos_val", val_idx)):
        eps = [episodes[i] for i in idx]
        save_transitions(out / f"{name}.npz",
                         np.concatenate([e["obs"] for e in eps]),
                         np.concatenate([e["actions"] for e in eps]),
                         np.concatenate([np.full(len(e["actions"]), e["episode_id"]) for e in eps]),
                         dict(source, split=name, episode_ids=[e["episode_id"] for e in eps]))
    raw = np.concatenate([e["raw_actions"] for e in episodes])
    summary = dict(
        state_dim=int(episodes[0]["obs"].shape[1]), action_dim=int(raw.shape[1]),
        episodes_in_file=len(meta["episodes"]), episodes_replayed=len(episodes),
        train_episodes=len(train_idx), val_episodes=len(val_idx),
        raw_action_min=raw.min(0).round(4).tolist(), raw_action_max=raw.max(0).round(4).tolist(),
        fraction_raw_actions_outside_bounds=float((np.abs(raw) > 1).mean()),
        replayed_success_at_end=float(np.mean([e["success_at_end"] for e in episodes])),
    )
    (out / "demos_summary.json").write_text(json.dumps(summary, indent=2))
    print(json.dumps(summary, indent=2))


# --------------------------------------------------------------------------------------
# Models
# --------------------------------------------------------------------------------------

class BCPolicy(nn.Module):
    """MLP teacher: normalized state -> action in [-1, 1].

    Normalization statistics are registered buffers, so a checkpoint carries its own
    preprocessing and the policy accepts raw environment observations.
    """

    def __init__(self, state_dim, action_dim, state_mean, state_scale, hidden=256, depth=2, tanh_output=True):
        super().__init__()
        layers, width = [], state_dim
        for _ in range(depth):
            layers += [nn.Linear(width, hidden), nn.ReLU()]
            width = hidden
        layers.append(nn.Linear(width, action_dim))
        self.net = nn.Sequential(*layers)
        self.tanh_output = tanh_output
        self.register_buffer("state_mean", torch.as_tensor(state_mean, dtype=torch.float32))
        self.register_buffer("state_scale", torch.as_tensor(state_scale, dtype=torch.float32))
        self.config = dict(state_dim=state_dim, action_dim=action_dim, hidden=hidden, depth=depth, tanh_output=tanh_output)

    def forward(self, raw_state):
        out = self.net((raw_state - self.state_mean) / self.state_scale)
        return torch.tanh(out) if self.tanh_output else out

    @torch.no_grad()
    def act(self, raw_state):
        return self(raw_state).clamp(-1.0, 1.0)


def save_policy(policy: BCPolicy, path, extra=None):
    torch.save({"config": policy.config, "state_dict": policy.state_dict(), "extra": extra or {}}, path)


def load_policy(path, device="cpu") -> BCPolicy:
    ckpt = torch.load(path, map_location=device, weights_only=False)
    cfg = ckpt["config"]
    policy = BCPolicy(cfg["state_dim"], cfg["action_dim"], np.zeros(cfg["state_dim"]), np.ones(cfg["state_dim"]),
                      hidden=cfg["hidden"], depth=cfg["depth"], tanh_output=cfg["tanh_output"])
    policy.load_state_dict(ckpt["state_dict"])
    return policy.to(device).eval()


class ReleasedPPOActor(nn.Module):
    """Deterministic mean of ManiSkill's state PPO actor (examples/baselines/ppo/ppo.py)."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.actor_mean = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(), nn.Linear(256, action_dim))

    @torch.no_grad()
    def act(self, raw_state):
        return self.actor_mean(raw_state).clamp(-1.0, 1.0)


def load_released_ppo(path, state_dim, action_dim, device="cpu"):
    weights = torch.load(path, map_location=device, weights_only=False)
    actor = ReleasedPPOActor(state_dim, action_dim)
    actor.load_state_dict({k: v for k, v in weights.items() if k.startswith("actor_mean.")})
    return actor.to(device).eval()


# --------------------------------------------------------------------------------------
# Rollouts (shared by evaluation and data collection)
# --------------------------------------------------------------------------------------

@torch.no_grad()
def rollout(env, max_steps, driver, seed, labeler=None, noise_std=0.0, generator=None):
    """Run one batch of episodes, one per parallel env, for max_steps with no resets.

    driver picks the executed action; Gaussian noise (std on the [-1, 1] action scale) is
    added after labeling, so labels stay the labeler's clean actions. If labeler is given,
    returns obs [n, T, d_s] and labels [n, T, d_a] on the CPU, episode-major.
    """
    obs, _ = env.reset(seed=seed)
    n = obs.shape[0]
    device = obs.device
    base = getattr(env, "base_env", None)
    if base is None:
        base = getattr(env, "unwrapped", None)   # raw ManiSkill env (no vector wrapper)
    # Per-episode start/goal records from the variable-goal envs, read right after reset.
    records = {name: getattr(base, name).clone() for name in EPISODE_RECORDS if hasattr(base, name)}
    if "goal_xy" not in records and hasattr(base, "goal_quadrant") and hasattr(base, "goal_tee"):
        records["goal_xy"] = base.goal_tee.pose.p[:, :2].clone()
    quadrant = records.get("goal_quadrant")
    success_once = torch.zeros(n, dtype=torch.bool, device=device)
    first_success = torch.full((n,), -1, dtype=torch.long, device=device)
    obs_buf, label_buf = [], []
    success = success_once
    for t in range(max_steps):
        o = obs.float()
        action = driver.act(o)
        if labeler is not None:
            label = action if labeler is driver else labeler.act(o)
            obs_buf.append(o.cpu())
            label_buf.append(label.cpu())
        if noise_std > 0:
            noise = torch.randn(action.shape, device=action.device, generator=generator)
            action = (action + noise_std * noise).clamp(-1.0, 1.0)
        obs, _, _, _, info = env.step(action)
        success = info["success"].bool()
        newly = (first_success < 0) & success
        first_success = torch.where(newly, torch.full_like(first_success, t + 1), first_success)
        success_once |= success
    steps = first_success[first_success > 0].float()
    result = {
        "episodes": int(n), "seed": int(seed), "noise_std": float(noise_std),
        "success_once": success_once.float().mean().item(),
        "success_at_end": success.float().mean().item(),
        "mean_steps_to_first_success": steps.mean().item() if len(steps) else None,
    }
    if quadrant is not None:
        result["by_goal_quadrant"] = {
            f"Q{q}": {"episodes": int(m.sum()),
                      "success_once": success_once[m].float().mean().item(),
                      "success_at_end": success[m].float().mean().item()}
            for q in range(1, 5) if (m := quadrant == q).any()}
    if "tip_ik_fallback" in records:
        result["tip_ik_fallback_rate"] = records["tip_ik_fallback"].float().mean().item()
    if labeler is not None:
        for name, value in records.items():
            result[f"{name}_per_episode"] = value.cpu().tolist()
    if labeler is None:
        return result
    return result, torch.stack(obs_buf, 1).numpy(), torch.stack(label_buf, 1).numpy()


# --------------------------------------------------------------------------------------
# Training
# --------------------------------------------------------------------------------------

@torch.no_grad()
def action_mse(policy, obs, actions, device, batch=65536):
    total, count = 0.0, 0
    for i in range(0, len(obs), batch):
        o = torch.as_tensor(obs[i:i + batch], device=device)
        a = torch.as_tensor(actions[i:i + batch], device=device)
        total += torch.nn.functional.mse_loss(policy(o), a, reduction="sum").item()
        count += a.numel()
    return total / max(count, 1)


def train(args):
    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = torch.device(args.device)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)

    obs_np, act_np, ep_np = load_transitions(args.train_data)
    val_obs, val_act, _ = load_transitions([args.val_data])
    mean, std, scale = compute_state_stats(obs_np)
    data_info = dict(train_files=[str(p) for p in args.train_data], val_file=str(args.val_data),
                     train_transitions=int(len(obs_np)), train_episodes=int(len(np.unique(ep_np))),
                     val_transitions=int(len(val_obs)),
                     features_only_centered=np.flatnonzero(std < MIN_SCALED_STD).tolist())
    (out / "data_info.json").write_text(json.dumps(data_info, indent=2))
    print(f"Training on {data_info['train_transitions']:,} transitions from "
          f"{data_info['train_episodes']:,} episodes in {len(args.train_data)} file(s)", flush=True)

    train_obs = torch.as_tensor(obs_np, device=device)
    train_act = torch.as_tensor(act_np, device=device).clamp(-1.0, 1.0)
    if args.tanh_output:
        # tanh cannot reach exactly +-1; keep targets just inside the bound.
        train_act = train_act.clamp(-0.999, 0.999)
    state_dim, action_dim = train_obs.shape[1], train_act.shape[1]
    policy = BCPolicy(state_dim, action_dim, mean, scale,
                      hidden=args.hidden, depth=args.depth, tanh_output=args.tanh_output).to(device)
    opt = torch.optim.Adam(policy.parameters(), lr=args.lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=args.iters) if args.cosine else None
    gen = torch.Generator(device=device).manual_seed(args.seed)

    evaluator = Evaluator(num_envs=args.eval_episodes, device=device) if args.eval_every > 0 else None

    fields = ("iter", "train_mse", "val_mse", "lr", "success_once", "success_at_end", "seconds")
    log = (out / "train_log.csv").open("w")
    log.write(",".join(fields) + "\n")
    best = (-1.0, -1.0)
    start = time.time()
    running, samples = 0.0, 0
    for it in range(1, args.iters + 1):
        idx = torch.randint(0, len(train_obs), (args.batch_size,), device=device, generator=gen)
        loss = torch.nn.functional.mse_loss(policy(train_obs[idx]), train_act[idx])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        if sched:
            sched.step()
        if it % 100 == 0:
            running += loss.item()
            samples += 1

        last = it == args.iters
        if it % args.log_every == 0 or last:
            policy.eval()
            val_mse = action_mse(policy, val_obs, val_act.clip(-1, 1), device)
            row = dict(iter=it, train_mse=running / max(samples, 1), val_mse=val_mse,
                       lr=opt.param_groups[0]["lr"], success_once="", success_at_end="")
            running, samples = 0.0, 0
            if evaluator and (it % args.eval_every == 0 or last):
                res = evaluator.run(policy, seed=args.val_seed)
                row.update(success_once=res["success_once"], success_at_end=res["success_at_end"])
                save_policy(policy, out / f"ckpt_{it}.pt", {"iter": it, **res})
                key = (res["success_at_end"], res["success_once"])
                if key > best:
                    best = key
                    save_policy(policy, out / "best.pt", {"iter": it, **res})
                    (out / "best.json").write_text(json.dumps({"iter": it, **res}, indent=2))
            policy.train()
            row["seconds"] = round(time.time() - start, 1)
            log.write(",".join(str(row[k]) for k in fields) + "\n")
            log.flush()
            msg = f"iter {it:>7}  train_mse {row['train_mse']:.5f}  val_mse {val_mse:.5f}"
            if row["success_once"] != "":
                msg += f"  val success_once {row['success_once']:.0%}  success_at_end {row['success_at_end']:.0%}"
            print(msg, flush=True)
    log.close()
    save_policy(policy, out / "final.pt", {"iter": args.iters})
    if evaluator is None:
        save_policy(policy, out / "best.pt", {"iter": args.iters, "note": "no closed-loop eval; best = final"})
    else:
        evaluator.close()
    print(f"Done in {(time.time() - start) / 60:.1f} min. Best validation (end, once): {best}")


# --------------------------------------------------------------------------------------
# Simulator: audit, evaluation, collection
# --------------------------------------------------------------------------------------

def configure_env(env_id, goal_quadrants=None, start_quadrants=None):
    """Select the environment for this process (registration happens in make_env)."""
    global ENV_ID, ENV_KWARGS
    ENV_ID, ENV_KWARGS = env_id, {}
    if goal_quadrants:
        if env_id not in GOAL_ENV_IDS:
            raise ValueError("--goal-quadrants only applies to " + " and ".join(GOAL_ENV_IDS))
        ENV_KWARGS["goal_quadrants"] = tuple(goal_quadrants)
    if start_quadrants:
        if env_id != VARIABLE_GOAL_ENV_ID:
            raise ValueError("--start-quadrants only applies to " + VARIABLE_GOAL_ENV_ID)
        ENV_KWARGS["start_quadrants"] = tuple(start_quadrants)


def make_env(num_envs, obs_mode=OBS_MODE, **kwargs):
    import gymnasium as gym
    import mani_skill.envs  # noqa: F401  (registers environments)

    if ENV_ID == VARIABLE_GOAL_ENV_ID:
        import variable_goal_env  # registers PushT-VariableGoal-v1
        if variable_goal_env.PushTVariableGoalEnv is None:
            raise ImportError("variable_goal_env could not import ManiSkill; the environment is not registered")
    elif ENV_ID == SHARED_ENV_ID:
        import pusht_vargoal  # noqa: F401  (registers PushTVarGoal-v1; Jimmy's state_rl/pusht_vargoal.py)
    return gym.make(ENV_ID, num_envs=num_envs, obs_mode=obs_mode, control_mode=CONTROL_MODE,
                    sim_backend=SIM_BACKEND, render_mode="rgb_array", **ENV_KWARGS, **kwargs)


def run_ppo(argv):
    """Run ManiSkill's ppo.py in this process after registering the variable-goal environment.
    ppo.py writes runs/<exp-name>/ under the current directory."""
    import runpy
    import sys

    p = argparse.ArgumentParser(prog="pusht_state_bc.py ppo", allow_abbrev=False)
    p.add_argument("--script", required=True, help="path to ManiSkill examples/baselines/ppo/ppo.py")
    known, rest = p.parse_known_args(argv)
    import variable_goal_env  # noqa: F401  (registers PushT-VariableGoal-v1 for ppo.py's gym.make)
    sys.argv = [known.script] + rest
    runpy.run_path(known.script, run_name="__main__")


def episode_length(env):
    """Max episode steps. Vectorized GPU envs leave env.spec.max_episode_steps as None,
    so use ManiSkill's own lookup (as its PPO baseline does), with PushT-v1's registered 100 as fallback."""
    from mani_skill.utils import gym_utils

    steps = gym_utils.find_max_episode_steps_value(env)
    return int(steps) if steps is not None else 100


def make_vector_env(num_envs, video_dir=None, **kwargs):
    """Fixed-length episodes: no auto-reset (it would replace the final step's info) and
    terminations ignored, matching the evaluation convention of ManiSkill's PPO baseline."""
    from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv

    env = make_env(num_envs, **kwargs)
    max_steps = episode_length(env)
    if video_dir is not None:
        from mani_skill.utils.wrappers.record import RecordEpisode
        env = RecordEpisode(env, output_dir=str(video_dir), save_trajectory=False,
                            max_steps_per_video=max_steps, video_fps=30)
    return ManiSkillVectorEnv(env, num_envs, auto_reset=False, ignore_terminations=True, record_metrics=True), max_steps


def flatten_with_names(tree, prefix=""):
    """Flatten a nested state dict in insertion order, as ManiSkill does; return names and widths."""
    fields = []
    for key, value in tree.items():
        name = f"{prefix}{key}"
        if isinstance(value, dict):
            fields += flatten_with_names(value, name + "/")
        else:
            width = 1 if value.ndim == 1 else int(value.shape[-1])
            fields.append((name, width))
    return fields


def audit(args):
    import mani_skill

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    env = make_env(1)
    try:
        obs, info = env.reset(seed=0)
        tree = env.unwrapped.get_obs(info, unflattened=True)
        fields = flatten_with_names(tree)
        width = sum(w for _, w in fields)
        assert obs.shape == (1, width), f"Flat obs {tuple(obs.shape)} does not match fields totalling {width}"
        offsets, start = {}, 0
        for name, w in fields:
            offsets[name] = [start, start + w]
            start += w
        if not any("obj_pose" in n for n, _ in fields):
            raise AssertionError("State observation lacks obj_pose; this is not the full-state setting")
        space = env.unwrapped.single_action_space
        contract = {
            "env_id": ENV_ID, "env_kwargs": {k: list(v) for k, v in ENV_KWARGS.items()}, "obs_mode": OBS_MODE, "control_mode": CONTROL_MODE, "sim_backend": SIM_BACKEND,
            "mani_skill_version": getattr(mani_skill, "__version__", "unknown"),
            "state_dim": width, "state_fields_in_order": offsets,
            "action_dim": int(np.prod(space.shape)),
            "action_low": space.low.tolist(), "action_high": space.high.tolist(),
            "control_freq": getattr(env.unwrapped, "control_freq", None),
            "sim_freq": getattr(env.unwrapped, "sim_freq", None),
            "max_episode_steps": episode_length(env),
            "robot_uids": str(getattr(env.unwrapped, "robot_uids", None)),
            "success_rule": "evaluate(): T overlap with goal >= intersection_thresh",
            "intersection_thresh": getattr(env.unwrapped, "intersection_thresh", None),
        }
        if not (np.allclose(space.low, -1) and np.allclose(space.high, 1)):
            print("WARNING: action space is not [-1, 1]; the BC tanh head assumes normalized actions.")
        (out / "observation_contract.json").write_text(json.dumps(contract, indent=2))
        print(json.dumps(contract, indent=2))
    finally:
        env.close()


def check_env(args):
    """Check the success test against each episode's own goal, without stepping physics.

    Puts a stand-in T exactly on every episode's goal pose and runs the environment's overlap
    computation: all must count as success. Then shifts it 5 cm: none may. With PushT-v1's single
    fixed goal transform, varied goals would fail the first check. Also reports goal ranges and quadrants.
    """
    import types

    env = make_env(args.num_envs)
    try:
        env.reset(seed=args.seed)
        base = env.unwrapped
        goal_p, goal_q = base.goal_tee.pose.p.clone(), base.goal_tee.pose.q.clone()
        real_tee = base.tee

        def overlap(p):
            base.tee = types.SimpleNamespace(pose=types.SimpleNamespace(p=p, q=goal_q))
            try:
                return base.pseudo_render_intersection()
            finally:
                base.tee = real_tee

        at_goal = overlap(goal_p)
        shifted = goal_p.clone()
        shifted[:, 0] += 0.05
        off_goal = overlap(shifted)
        thresh = base.intersection_thresh
        quadrant = getattr(base, "goal_quadrant", None)
        report = {
            "env_id": ENV_ID, "num_envs": args.num_envs,
            "goal_x_range": [goal_p[:, 0].min().item(), goal_p[:, 0].max().item()],
            "goal_y_range": [goal_p[:, 1].min().item(), goal_p[:, 1].max().item()],
            "goal_quadrant_counts": ({f"Q{q}": int((quadrant == q).sum()) for q in range(1, 5)}
                                     if quadrant is not None else None),
            "overlap_with_T_on_goal_min": at_goal.min().item(),
            "overlap_with_T_shifted_5cm_max": off_goal.max().item(),
            "intersection_thresh": thresh,
        }
        passed = bool((at_goal >= thresh).all() and (off_goal < thresh).all())
        if hasattr(base, "tip_start_xy") and getattr(base, "random_tip_start", False):
            # Where the stick tip actually is after reset, versus where IK placed it.
            tcp_xy = base.agent.tcp.pose.p[:, :2]
            placed = ~base.tip_ik_fallback
            gap = torch.linalg.norm(tcp_xy - base.tip_start_xy, dim=1)
            tee_dist = torch.linalg.norm(tcp_xy - base.tee.pose.p[:, :2], dim=1)
            tip_q = base.tip_start_quadrant
            report.update({
                "tip_x_range": [tcp_xy[:, 0].min().item(), tcp_xy[:, 0].max().item()],
                "tip_y_range": [tcp_xy[:, 1].min().item(), tcp_xy[:, 1].max().item()],
                "tip_start_quadrant_counts": {f"Q{q}": int((tip_q == q).sum()) for q in range(1, 5)},
                "tee_start_quadrant_counts": {f"Q{q}": int((base.tee_start_quadrant == q).sum()) for q in range(1, 5)},
                "tip_ik_fallback_rate": base.tip_ik_fallback.float().mean().item(),
                "tip_grid_reachable_fraction": base._ik["valid_fraction"] if getattr(base, "_ik", None) else None,
                "tip_ik_error_max_m": base.tip_ik_error.max().item(),
                "tip_vs_ik_gap_max_m": gap[placed].max().item() if placed.any() else None,
                "tip_height_range_m": [base.agent.tcp.pose.p[:, 2].min().item(), base.agent.tcp.pose.p[:, 2].max().item()],
                "tip_to_tee_min_m": tee_dist.min().item(),
                "goal_vs_tip_quadrant_agreement": (tip_q == quadrant).float().mean().item() if quadrant is not None else None,
            })
            # Placement must match the simulator's own kinematics (catches a wrong base offset or chain)
            # and fall back rarely.
            passed = passed and report["tip_ik_fallback_rate"] <= 0.05 and (
                report["tip_vs_ik_gap_max_m"] is None or report["tip_vs_ik_gap_max_m"] < 0.02)
        report["passed"] = passed
    finally:
        env.close()
    print(json.dumps(report, indent=2))
    if args.out:
        Path(args.out).write_text(json.dumps(report, indent=2))
    if not report["passed"]:
        raise SystemExit("Environment check FAILED: success is not measured against each episode's goal.")


class Evaluator:
    """Closed-loop evaluation on a fixed batch of seeded initial states."""

    def __init__(self, num_envs, device, video_dir=None):
        self.env, self.max_steps = make_vector_env(num_envs, video_dir=video_dir, reconfiguration_freq=1)
        self.num_envs = num_envs
        self.device = device

    def run(self, policy, seed):
        return rollout(self.env, self.max_steps, policy, seed)

    def close(self):
        self.env.close()


def load_expert(path, env, device):
    spaces = env if hasattr(env, "single_observation_space") else env.unwrapped
    state_dim = int(np.prod(spaces.single_observation_space.shape))
    action_dim = int(np.prod(spaces.single_action_space.shape))
    return load_released_ppo(path, state_dim, action_dim, device)


def eval_per_quadrant(env, max_steps, policy, base_seed, quadrants=(1, 2, 3, 4), bar=None):
    """Shared per-quadrant protocol (state_rl/pusht_state_ppo.py): for each quadrant q, restrict goals
    to q and run one batch of episodes with seed base_seed + 1000 * q."""
    base = env.unwrapped
    original = tuple(base.goal_quadrants)
    out = {}
    try:
        for q in quadrants:
            base.goal_quadrants = (q,)
            r = rollout(env, max_steps, policy, base_seed + 1000 * q)
            if not bool((base.goal_quadrant == q).all()):
                raise AssertionError(f"goals left the requested quadrant Q{q}")
            r.pop("by_goal_quadrant", None)
            out[f"Q{q}"] = r
    finally:
        base.goal_quadrants = original
    once = [out[f"Q{q}"]["success_once"] for q in quadrants]
    end = [out[f"Q{q}"]["success_at_end"] for q in quadrants]
    out.update(worst_success_once=min(once), mean_success_once=float(np.mean(once)),
               mean_success_at_end=float(np.mean(end)))
    if bar is not None:
        out.update(freeze_bar=bar, passes_bar=all(v >= bar for v in once))
    return out


def eval_quadrants(args):
    """Each checkpoint, quadrant by quadrant, on a raw (unwrapped, not reconfigured) env, as in
    state_rl/pusht_state_ppo.py, so BC and PPO teachers are scored on the same episodes."""
    device = torch.device(args.device)
    env = make_env(args.episodes)
    if not hasattr(env.unwrapped, "goal_quadrants"):
        env.close()
        raise SystemExit(f"{ENV_ID} has no goal_quadrants; eval-quadrants needs a variable-goal env")
    rows = []
    try:
        max_steps = episode_length(env)
        for ckpt in args.checkpoint:
            policy = load_expert(ckpt, env, device) if args.ppo else load_policy(ckpt, device)
            r = eval_per_quadrant(env, max_steps, policy, args.seed, bar=args.bar)
            rows.append({"checkpoint": str(ckpt), "env_id": ENV_ID, "episodes_per_quadrant": args.episodes,
                         "base_seed": args.seed, **r})
            print(f"{Path(ckpt).name:>24}  worst {r['worst_success_once']:.0%}  mean {r['mean_success_once']:.0%}  "
                  + "  ".join(f"Q{q} {r[f'Q{q}']['success_once']:.0%}/{r[f'Q{q}']['success_at_end']:.0%}"
                              for q in (1, 2, 3, 4)), flush=True)
    finally:
        env.close()
    if args.out:
        Path(args.out).write_text(json.dumps(rows if len(rows) > 1 else rows[0], indent=2))
    return rows


def evaluate(args):
    device = torch.device(args.device)
    video_dir = Path(args.video_dir) if args.video_dir else None
    evaluator = Evaluator(args.episodes, device, video_dir=video_dir)
    try:
        if args.ppo:
            policy = load_expert(args.checkpoint, evaluator.env, device)
        else:
            policy = load_policy(args.checkpoint, device)
        res = evaluator.run(policy, seed=args.seed)
    finally:
        evaluator.close()
    res["checkpoint"] = str(args.checkpoint)
    print(json.dumps(res, indent=2))
    if args.out:
        Path(args.out).write_text(json.dumps(res, indent=2))
    return res


def collect(args):
    """Roll out a driver and label every visited state with the expert.

    driver=expert (optionally with action noise) gives more, and slightly off-course,
    expert demonstrations. driver=bc gives one round of DAgger data: states the BC policy
    actually reaches, labeled with what the expert would do there.
    """
    device = torch.device(args.device)
    env, max_steps = make_vector_env(args.num_envs)
    try:
        expert = load_expert(args.expert, env, device)
        driver = expert if args.driver == "expert" else load_policy(args.driver_checkpoint, device)
        noise_levels = args.noise_std or [0.0]
        gen = torch.Generator(device=device).manual_seed(args.seed_base)
        batches = math.ceil(args.episodes / args.num_envs)
        all_obs, all_labels, per_batch, all_noise = [], [], [], []
        per_episode = {}
        for b in range(batches):
            sigma = noise_levels[b % len(noise_levels)]
            res, obs, labels = rollout(env, max_steps, driver, seed=args.seed_base + b,
                                       labeler=expert, noise_std=sigma, generator=gen)
            all_obs.append(obs)
            all_labels.append(labels)
            for key in [k for k in res if k.endswith("_per_episode")]:
                per_episode.setdefault(key, []).extend(res.pop(key))
            all_noise += [sigma] * obs.shape[0]
            per_batch.append(res)
            print(f"batch {b + 1}/{batches}  noise {sigma:.2f}  driver success_once {res['success_once']:.0%}  "
                  f"success_at_end {res['success_at_end']:.0%}", flush=True)
    finally:
        env.close()
    obs = np.concatenate(all_obs)[:args.episodes]       # [episodes, T, d_s]
    labels = np.concatenate(all_labels)[:args.episodes]
    n, T = obs.shape[:2]
    meta = dict(
        driver=args.driver, driver_checkpoint=args.driver_checkpoint or None, expert=str(args.expert),
        labels="expert deterministic action, clipped to [-1, 1]", noise_std_cycle=noise_levels,
        env_id=ENV_ID, env_kwargs={k: list(v) for k, v in ENV_KWARGS.items()},
        seed_base=args.seed_base, num_envs=args.num_envs, steps_per_episode=int(T), batches=per_batch,
        driver_success_once=float(np.mean([r["success_once"] for r in per_batch])),
        driver_success_at_end=float(np.mean([r["success_at_end"] for r in per_batch])),
        noise_std_per_episode=all_noise[:n],
    )
    for key, values in per_episode.items():
        meta[key] = values[:n]
    meta = save_transitions(args.out, obs.reshape(n * T, -1), labels.reshape(n * T, -1),
                            np.repeat(np.arange(n), T), meta)
    print(json.dumps({k: v for k, v in meta.items() if not k.endswith("per_episode") and k != "batches"}, indent=2))


# --------------------------------------------------------------------------------------
# CLI
# --------------------------------------------------------------------------------------

def main(argv=None):
    import sys

    argv = sys.argv[1:] if argv is None else list(argv)
    if argv and argv[0] == "ppo":
        return run_ppo(argv[1:])

    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    sub.add_parser("ppo", help="run ManiSkill ppo.py on the variable-goal env (see run_ppo)")
    default_device = "cuda" if torch.cuda.is_available() else "cpu"
    env_opts = argparse.ArgumentParser(add_help=False)
    env_opts.add_argument("--env-id", default=VARIABLE_GOAL_ENV_ID)
    env_opts.add_argument("--goal-quadrants", type=int, nargs="+", default=None,
                          help="goal quadrants to sample (variable-goal env only); default all four")
    env_opts.add_argument("--start-quadrants", type=int, nargs="+", default=None,
                          help="stick-tip start quadrants to sample (variable-goal env only); default all four")

    a = sub.add_parser("audit", parents=[env_opts])
    a.add_argument("--out", required=True)

    ce = sub.add_parser("check-env", parents=[env_opts])
    ce.add_argument("--num-envs", type=int, default=256)
    ce.add_argument("--seed", type=int, default=0)
    ce.add_argument("--out", default="")

    pr = sub.add_parser("prepare")
    pr.add_argument("--h5", required=True)
    pr.add_argument("--out-dir", required=True)
    pr.add_argument("--val-fraction", type=float, default=0.1)
    pr.add_argument("--split-seed", type=int, default=0)
    pr.add_argument("--max-episodes", type=int, default=0, help="0 = use all episodes")

    c = sub.add_parser("collect", parents=[env_opts])
    c.add_argument("--expert", required=True, help="PPO checkpoint (ManiSkill ppo.py format) used for labels")
    c.add_argument("--driver", choices=("expert", "bc"), default="expert")
    c.add_argument("--driver-checkpoint", default="", help="BC checkpoint when --driver bc")
    c.add_argument("--episodes", type=int, required=True)
    c.add_argument("--num-envs", type=int, default=1024)
    c.add_argument("--noise-std", type=float, nargs="*", default=[0.0],
                   help="action noise std per batch, cycled; labels stay noise-free")
    c.add_argument("--seed-base", type=int, required=True)
    c.add_argument("--out", required=True)
    c.add_argument("--device", default=default_device)

    t = sub.add_parser("train", parents=[env_opts])
    t.add_argument("--train-data", nargs="+", required=True)
    t.add_argument("--val-data", required=True)
    t.add_argument("--out", required=True)
    t.add_argument("--seed", type=int, default=0)
    t.add_argument("--iters", type=int, default=100_000)
    t.add_argument("--batch-size", type=int, default=1024)
    t.add_argument("--lr", type=float, default=3e-4)
    t.add_argument("--cosine", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--hidden", type=int, default=256)
    t.add_argument("--depth", type=int, default=2)
    t.add_argument("--tanh-output", action=argparse.BooleanOptionalAction, default=True)
    t.add_argument("--log-every", type=int, default=1000)
    t.add_argument("--eval-every", type=int, default=10_000, help="0 disables closed-loop eval")
    t.add_argument("--eval-episodes", type=int, default=100)
    t.add_argument("--val-seed", type=int, default=1001)
    t.add_argument("--device", default=default_device)

    eq = sub.add_parser("eval-quadrants", parents=[env_opts])
    eq.add_argument("--checkpoint", nargs="+", required=True)
    eq.add_argument("--ppo", action="store_true", help="checkpoints are ManiSkill ppo.py checkpoints")
    eq.add_argument("--episodes", type=int, default=100, help="per quadrant")
    eq.add_argument("--seed", type=int, default=1001, help="quadrant q uses seed + 1000 * q")
    eq.add_argument("--bar", type=float, default=0.80, help="freeze bar: success-once in every quadrant")
    eq.add_argument("--out", default="")
    eq.add_argument("--device", default=default_device)

    e = sub.add_parser("eval", parents=[env_opts])
    e.add_argument("--checkpoint", required=True)
    e.add_argument("--ppo", "--released-ppo", dest="ppo", action="store_true",
                   help="checkpoint is a ManiSkill ppo.py checkpoint (released or our own)")
    e.add_argument("--episodes", type=int, default=100)
    e.add_argument("--seed", type=int, default=2001)
    e.add_argument("--out", default="")
    e.add_argument("--video-dir", default="")
    e.add_argument("--device", default=default_device)

    args = p.parse_args(argv)
    if args.cmd == "collect" and args.driver == "bc" and not args.driver_checkpoint:
        p.error("--driver bc needs --driver-checkpoint")
    if hasattr(args, "env_id"):
        configure_env(args.env_id, args.goal_quadrants, args.start_quadrants)
    {"audit": audit, "check-env": check_env, "prepare": prepare, "collect": collect, "train": train,
     "eval": evaluate, "eval-quadrants": eval_quadrants}[args.cmd](args)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /content/pusht_state_bc/pusht_vargoal.py
# Copied unchanged from jimmy-state-rl:state_rl/pusht_vargoal.py at c87d8ae
"""Variable-goal Push-T: ManiSkill PushT-v1 with a different target position per episode.

Registers ``PushTVarGoal-v1``. Import this module before ``gym.make``.

What changes relative to PushT-v1 (ManiSkill 3.0.1, commit a4a4f92):
  * Goal T position is sampled per episode from four quadrants around ``QUAD_ORIGIN``.
    Goal orientation stays fixed at PushT-v1's 5*pi/3.
  * The success check (``pseudo_render_intersection``) uses a goal transform per
    environment, recomputed at every reset. In PushT-v1 it is one fixed matrix, so a
    moved goal would silently be scored against the old target.
  * The T block starts anywhere in ``TEE_BOX`` (independent of the goal quadrant), with
    its distance to the goal limited to ``START_GOAL_DIST``.
  * The stick tip starts at a random point in ``TIP_BOX``, drawn from a precomputed
    library of arm poses (``assets/tip_start_qpos.npz``, built by make_tip_starts.py),
    at least ``TIP_TEE_MIN`` from the block.

What does NOT change: observation (31 numbers, same order; ``extra/goal_pos`` now
varies), action space and controller, 100-step episodes, the dense reward (it already
reads the goal T's live pose), and the success rule (overlap >= 0.90).

Quadrants, with dx = x - QUAD_ORIGIN[0] (positive = away from the robot base) and y:
  Q1: dx > 0, y > 0    Q2: dx < 0, y > 0    Q3: dx < 0, y < 0    Q4: dx > 0, y < 0
A band of ``AXIS_GAP`` around each axis is excluded so neighbouring quadrants do not
share near-identical goals.

Usage:
  env = gym.make("PushTVarGoal-v1", obs_mode="state", control_mode="pd_joint_delta_pos",
                 goal_quadrants=(1, 2, 3, 4))
  env.reset(seed=0)                                    # random goals in those quadrants
  env.reset(options=dict(goal_xy=torch.tensor([[-0.2, 0.1]])))  # fixed goal(s), [b, 2]
  env.unwrapped.goal_quadrant                          # [num_envs] quadrant of current goal
"""
from pathlib import Path

import numpy as np
import torch

from mani_skill.envs.tasks.tabletop.push_t import PushTEnv
from mani_skill.utils.registration import register_env
from mani_skill.utils.structs.pose import Pose

# ---- Workspace (world frame, metres). Robot base is at x = -0.615. ----------------
# Measured stick-tip reach at table height: x in about [-0.40, 0.05], y in about
# [-0.45, 0.45] (see README). Regions below keep >= ~0.1 m margin inside that.
QUAD_ORIGIN = (-0.156, 0.0)          # PushT-v1's goal x; centred in y
GOAL_HALF = (0.10, 0.20)             # goals in origin +- these
AXIS_GAP = (0.02, 0.03)              # excluded band around each quadrant axis
TEE_BOX = ((-0.30, -0.02), (-0.28, 0.28))   # T centre-of-mass start region
TIP_BOX = ((-0.36, 0.00), (-0.30, 0.30))    # stick-tip start region
START_GOAL_DIST = (0.08, 0.30)       # allowed T-start-to-goal distance
TIP_TEE_MIN = 0.15                   # stick tip starts at least this far from the T
N_CANDIDATES = 64                    # rejection-sampling candidates per reset

_QUAD_SIGNS = {1: (1.0, 1.0), 2: (-1.0, 1.0), 3: (-1.0, -1.0), 4: (1.0, -1.0)}
_DEFAULT_TIP_FILE = Path(__file__).parent / "assets" / "tip_start_qpos.npz"


def quadrant_of(goal_xy: torch.Tensor) -> torch.Tensor:
    """Quadrant (1-4) of goal positions [b, 2]."""
    dx = goal_xy[:, 0] - QUAD_ORIGIN[0]
    dy = goal_xy[:, 1] - QUAD_ORIGIN[1]
    q = torch.full_like(dx, 0, dtype=torch.long)
    q[(dx > 0) & (dy > 0)] = 1
    q[(dx < 0) & (dy > 0)] = 2
    q[(dx < 0) & (dy < 0)] = 3
    q[(dx > 0) & (dy < 0)] = 4
    return q


def sample_goals(b: int, quadrants, device=None) -> torch.Tensor:
    """Uniform goal positions [b, 2] in the given quadrants (each quadrant equally likely)."""
    quads = torch.tensor(list(quadrants), device=device)
    pick = quads[torch.randint(len(quads), (b,), device=device)]
    signs = torch.tensor([_QUAD_SIGNS[int(k)] for k in range(1, 5)], device=device)[pick - 1]
    lo = torch.tensor(AXIS_GAP, device=device)
    hi = torch.tensor(GOAL_HALF, device=device)
    mag = lo + torch.rand(b, 2, device=device) * (hi - lo)
    return torch.tensor(QUAD_ORIGIN, device=device) + signs * mag


def _uniform_box(shape, box, device):
    lo = torch.tensor([box[0][0], box[1][0]], device=device)
    hi = torch.tensor([box[0][1], box[1][1]], device=device)
    return lo + torch.rand(*shape, 2, device=device) * (hi - lo)


def _first_valid(valid: torch.Tensor) -> torch.Tensor:
    """Index of the first True per row of [b, K]; 0 if none (vanishingly rare with K=64)."""
    return valid.float().argmax(dim=1)


@register_env("PushTVarGoal-v1", max_episode_steps=100)
class PushTVarGoalEnv(PushTEnv):
    def __init__(self, *args, goal_quadrants=(1, 2, 3, 4), tip_start_file=None, **kwargs):
        assert set(goal_quadrants) <= {1, 2, 3, 4} and len(goal_quadrants) > 0
        self.goal_quadrants = tuple(int(q) for q in goal_quadrants)
        lib = np.load(tip_start_file or _DEFAULT_TIP_FILE)
        self._tip_qpos_np = lib["qpos"]
        self._tip_xy_np = lib["tip_xy"]
        super().__init__(*args, **kwargs)

    def _load_scene(self, options: dict):
        super()._load_scene(options)
        # One goal transform per environment instead of PushT-v1's single matrix.
        self.world_to_goal_trans = self.world_to_goal_trans.unsqueeze(0).repeat(self.num_envs, 1, 1)
        self.goal_quadrant = torch.zeros(self.num_envs, dtype=torch.long, device=self.device)
        self.tip_qpos_lib = torch.as_tensor(self._tip_qpos_np, device=self.device)
        self.tip_xy_lib = torch.as_tensor(self._tip_xy_np, device=self.device)

    def _goal_world_to_goal(self, goal_xy: torch.Tensor) -> torch.Tensor:
        b = goal_xy.shape[0]
        c, s = np.cos(self.goal_z_rot), np.sin(self.goal_z_rot)
        g = torch.zeros(b, 3, 3, device=goal_xy.device)
        g[:, 0, 0], g[:, 0, 1], g[:, 1, 0], g[:, 1, 1] = c, -s, s, c
        g[:, 2, 2] = 1.0
        g[:, 0:2, 2] = goal_xy
        return torch.linalg.inv(g)

    def _initialize_episode(self, env_idx: torch.Tensor, options: dict):
        # Parent sets up the table, default goal/T and robot rest pose; we override all three.
        super()._initialize_episode(env_idx, options)
        with torch.device(self.device):
            b = len(env_idx)

            # ---- goal ----
            goal_xy = (options or {}).get("goal_xy")
            if goal_xy is None:
                goal_xy = sample_goals(b, self.goal_quadrants, self.device)
            else:
                goal_xy = torch.as_tensor(goal_xy, dtype=torch.float32, device=self.device).reshape(-1, 2)
                if goal_xy.shape[0] == 1 and b > 1:
                    goal_xy = goal_xy.expand(b, 2)
                elif goal_xy.shape[0] == self.num_envs and b < self.num_envs:
                    goal_xy = goal_xy[env_idx]
                assert goal_xy.shape[0] == b, "goal_xy must be [1, 2], [b, 2] or [num_envs, 2]"
            goal_p = torch.zeros(b, 3)
            goal_p[:, :2] = goal_xy
            goal_p[:, 2] = 1e-3
            goal_q = torch.tensor([np.cos(self.goal_z_rot / 2), 0.0, 0.0, np.sin(self.goal_z_rot / 2)],
                                  dtype=torch.float32).expand(b, 4)
            self.goal_tee.set_pose(Pose.create_from_pq(p=goal_p, q=goal_q))
            self.world_to_goal_trans[env_idx] = self._goal_world_to_goal(goal_xy)
            self.goal_quadrant[env_idx] = quadrant_of(goal_xy)

            # ---- T block start: independent of goal quadrant, distance-limited ----
            cand = _uniform_box((b, N_CANDIDATES), TEE_BOX, self.device)          # [b, K, 2]
            d = (cand - goal_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid((d >= START_GOAL_DIST[0]) & (d <= START_GOAL_DIST[1]))
            tee_xy = cand[torch.arange(b), pick]
            tee_p = torch.zeros(b, 3)
            tee_p[:, :2] = tee_xy
            tee_p[:, 2] = 0.04 / 2 + 1e-3
            ang = torch.rand(b) * (2 * torch.pi)
            tee_q = torch.zeros(b, 4)
            tee_q[:, 0], tee_q[:, 3] = (ang / 2).cos(), (ang / 2).sin()
            self.tee.set_pose(Pose.create_from_pq(p=tee_p, q=tee_q))

            # ---- stick tip start: library pose away from the T ----
            n_lib = self.tip_xy_lib.shape[0]
            idx = torch.randint(n_lib, (b, N_CANDIDATES))
            dt = (self.tip_xy_lib[idx] - tee_xy[:, None, :]).norm(dim=-1)
            pick = _first_valid(dt >= TIP_TEE_MIN)
            qpos = self.tip_qpos_lib[idx[torch.arange(b), pick]]
            qpos = qpos + torch.randn_like(qpos) * self.robot_init_qpos_noise
            self.agent.reset(qpos)


In [ ]:
%%writefile /content/pusht_state_bc/pusht_state_ppo.py
# Copied unchanged from jimmy-state-rl:state_rl/pusht_state_ppo.py at c87d8ae
"""Goal-conditioned state PPO teacher for PushTVarGoal-v1 (Jimmy).

Training itself is ManiSkill's unchanged PPO baseline (examples/baselines/ppo/ppo.py at
commit a4a4f92), launched through run_ppo.py so the variable-goal env is registered.
This module holds everything around it:

  preflight  GPU checks of the variable-goal env (partial resets, per-env success check),
             plus observation_contract.json
  eval       per-quadrant closed-loop evaluation of PPO checkpoints, deterministic mean
             action clipped to [-1, 1]; selects a checkpoint on validation seeds and
             reports whether it clears the freeze bar in every quadrant

Metrics follow state_bc: success_once (reached the goal at some step) and success_at_end
(at the goal at step 100); episodes run the full 100 steps with no early reset.
Evaluation seeds: episodes for quadrant q use seed (base_seed + 1000 * q).
"""
import argparse
import json
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn

sys.path.insert(0, str(Path(__file__).resolve().parent))
import pusht_vargoal as pv  # noqa: E402  (registers PushTVarGoal-v1)

ENV_ID = "PushTVarGoal-v1"
CONTROL_MODE = "pd_joint_delta_pos"
OBS_FIELDS = [("agent/qpos", 7), ("agent/qvel", 7), ("extra/tcp_pose", 7),
              ("extra/goal_pos", 3), ("extra/obj_pose", 7)]  # same order as state_bc


def make_env(num_envs, sim_backend="physx_cuda", quadrants=(1, 2, 3, 4), **kw):
    import gymnasium as gym
    return gym.make(ENV_ID, num_envs=num_envs, obs_mode="state", control_mode=CONTROL_MODE,
                    sim_backend=sim_backend, goal_quadrants=quadrants, **kw)


# --------------------------------------------------------------------------------------
# Policy: deterministic mean of ManiSkill's state PPO actor
# --------------------------------------------------------------------------------------

class PPOActor(nn.Module):
    """actor_mean of examples/baselines/ppo/ppo.py (3 x 256 tanh)."""

    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.actor_mean = nn.Sequential(
            nn.Linear(state_dim, 256), nn.Tanh(), nn.Linear(256, 256), nn.Tanh(),
            nn.Linear(256, 256), nn.Tanh(), nn.Linear(256, action_dim))

    @torch.no_grad()
    def act(self, raw_state):
        return self.actor_mean(raw_state).clamp(-1.0, 1.0)


def load_actor(path, state_dim=31, action_dim=7, device="cpu"):
    weights = torch.load(path, map_location=device, weights_only=False)
    actor = PPOActor(state_dim, action_dim)
    actor.load_state_dict({k: v for k, v in weights.items() if k.startswith("actor_mean.")})
    return actor.to(device).eval()


# --------------------------------------------------------------------------------------
# Evaluation
# --------------------------------------------------------------------------------------

@torch.no_grad()
def rollout(env, actor, seed, max_steps=100):
    """One batch of episodes (one per env) for max_steps with no resets."""
    obs, _ = env.reset(seed=seed)
    n = obs.shape[0]
    dev = obs.device
    success_once = torch.zeros(n, dtype=torch.bool, device=dev)
    first = torch.full((n,), -1, dtype=torch.long, device=dev)
    success = success_once
    for t in range(max_steps):
        obs, _, _, _, info = env.step(actor.act(obs.float()))
        success = info["success"].bool()
        first = torch.where((first < 0) & success, torch.full_like(first, t + 1), first)
        success_once |= success
    steps = first[first > 0].float()
    return {"episodes": int(n), "seed": int(seed),
            "success_once": success_once.float().mean().item(),
            "success_at_end": success.float().mean().item(),
            "mean_steps_to_first_success": steps.mean().item() if len(steps) else None}


def eval_per_quadrant(env, actor, base_seed, quadrants=(1, 2, 3, 4), max_steps=100):
    u = env.unwrapped
    out = {}
    for q in quadrants:
        u.goal_quadrants = (q,)
        r = rollout(env, actor, base_seed + 1000 * q, max_steps)
        assert bool((u.goal_quadrant == q).all()), "goals left the requested quadrant"
        out[f"Q{q}"] = r
    u.goal_quadrants = (1, 2, 3, 4)
    so = [out[f"Q{q}"]["success_once"] for q in quadrants]
    se = [out[f"Q{q}"]["success_at_end"] for q in quadrants]
    out["worst_success_once"] = min(so)
    out["mean_success_once"] = float(np.mean(so))
    out["mean_success_at_end"] = float(np.mean(se))
    return out


def checkpoint_iteration(p: Path):
    s = p.stem.split("_")[-1]
    return int(s) if s.isdigit() else 10**9  # final_ckpt sorts last


def cmd_eval(args):
    device = torch.device(args.device)
    ckpts = sorted(Path(args.run_dir).glob("*.pt"), key=checkpoint_iteration) if args.run_dir else []
    ckpts = ckpts[-args.last:] if args.last else ckpts
    ckpts += [Path(c) for c in args.ckpt]
    assert ckpts, "no checkpoints found"
    env = make_env(args.episodes, sim_backend=args.sim_backend)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    rows = []
    for c in ckpts:
        actor = load_actor(c, device=device)
        r = eval_per_quadrant(env, actor, args.val_seed)
        rows.append({"checkpoint": str(c), **r})
        print(f"{c.name:>18}  worst {r['worst_success_once']:.0%}  mean {r['mean_success_once']:.0%}  "
              + "  ".join(f"Q{q} {r[f'Q{q}']['success_once']:.0%}/{r[f'Q{q}']['success_at_end']:.0%}"
                          for q in (1, 2, 3, 4)), flush=True)
    (out / "val_results.json").write_text(json.dumps(rows, indent=2))
    # Selection on validation only: best worst-quadrant success_once, then mean, then later checkpoint.
    best = max(rows, key=lambda r: (r["worst_success_once"], r["mean_success_once"],
                                    checkpoint_iteration(Path(r["checkpoint"]))))
    print("\nSelected on validation:", best["checkpoint"])
    result = {"selected": best["checkpoint"], "val": best, "freeze_bar": args.bar,
              "val_seed": args.val_seed, "episodes_per_quadrant": args.episodes}
    if args.test_seed is not None:
        test = eval_per_quadrant(env, load_actor(best["checkpoint"], device=device), args.test_seed)
        result.update(test=test, test_seed=args.test_seed,
                      passes_bar=all(test[f"Q{q}"]["success_once"] >= args.bar for q in (1, 2, 3, 4)))
        print("Test (used once):  " + "  ".join(
            f"Q{q} {test[f'Q{q}']['success_once']:.0%}/{test[f'Q{q}']['success_at_end']:.0%}" for q in (1, 2, 3, 4)))
        print(f"Freeze bar {args.bar:.0%} success_once in every quadrant:",
              "PASS" if result["passes_bar"] else "FAIL")
    (out / "selection.json").write_text(json.dumps(result, indent=2))
    env.close()


# --------------------------------------------------------------------------------------
# Preflight
# --------------------------------------------------------------------------------------

def _place_tee(u, xy, ang):
    from mani_skill.utils.structs.pose import Pose
    b = xy.shape[0]
    p = torch.zeros(b, 3, device=xy.device)
    p[:, :2], p[:, 2] = xy, 0.021
    q = torch.zeros(b, 4, device=xy.device)
    q[:, 0], q[:, 3] = np.cos(ang / 2), np.sin(ang / 2)
    u.tee.set_pose(Pose.create_from_pq(p=p, q=q))


def preflight(num_envs=8, sim_backend="physx_cuda", out=None):
    env = make_env(num_envs, sim_backend=sim_backend)
    u = env.unwrapped
    obs, _ = env.reset(seed=0)
    from mani_skill.utils import gym_utils
    checks = {}

    # 1. observation layout and spaces match state_bc's contract
    raw = u._get_obs_agent()
    extra = u._get_obs_extra(u.get_info())
    parts = [raw["qpos"], raw["qvel"], extra["tcp_pose"], extra["goal_pos"], extra["obj_pose"]]
    checks["obs_layout"] = bool(torch.allclose(torch.cat(parts, -1).float(), obs.float(), atol=1e-5))
    checks["dims"] = tuple(obs.shape[1:]) == (31,) and tuple(u.single_action_space.shape) == (7,)
    checks["max_episode_steps_100"] = gym_utils.find_max_episode_steps_value(env) == 100

    # 2. every env scores success against its own goal
    goals = u.goal_tee.pose.p[:, :2].clone()
    _place_tee(u, goals, u.goal_z_rot)
    checks["success_at_own_goal"] = bool(u.evaluate()["success"].all())
    shifted = goals.roll(1, 0)
    far = (shifted - goals).norm(dim=1) > 0.04
    _place_tee(u, shifted, u.goal_z_rot)
    checks["no_success_at_other_goal"] = bool((~u.evaluate()["success"][far]).all()) if far.any() else True

    # 3. partial reset changes only the reset envs (what PPO's auto-reset does)
    if num_envs >= 2:
        env.reset(seed=1)
        before_goal = u.goal_tee.pose.p[:, :2].clone()
        before_trans = u.world_to_goal_trans.clone()
        idx = torch.arange(0, num_envs, 2, device=u.device)
        keep = torch.arange(1, num_envs, 2, device=u.device)
        env.reset(options=dict(env_idx=idx))
        after_goal = u.goal_tee.pose.p[:, :2]
        checks["partial_reset_keeps_others"] = bool(
            torch.allclose(after_goal[keep], before_goal[keep]) and
            torch.allclose(u.world_to_goal_trans[keep], before_trans[keep]))
        checks["partial_reset_changes_reset_envs"] = bool((after_goal[idx] - before_goal[idx]).norm(dim=1).gt(1e-4).all())
        _place_tee(u, after_goal.clone(), u.goal_z_rot)
        checks["success_after_partial_reset"] = bool(u.evaluate()["success"].all())
        checks["quadrant_labels_after_partial_reset"] = bool((u.goal_quadrant == pv.quadrant_of(after_goal)).all())

    # 4. quadrant restriction
    u.goal_quadrants = (4,)
    env.reset(seed=2)
    checks["quadrant_restriction"] = bool((u.goal_quadrant == 4).all())
    u.goal_quadrants = (1, 2, 3, 4)

    contract = {
        "env_id": ENV_ID, "base_env": "PushT-v1", "obs_mode": "state", "control_mode": CONTROL_MODE,
        "state_dim": 31, "action_dim": 7, "max_episode_steps": 100,
        "control_freq": u.control_freq, "sim_freq": u.sim_freq, "sim_backend": sim_backend,
        "state_fields": [{"name": n, "start": s, "stop": s + k}
                         for (n, k), s in zip(OBS_FIELDS, np.cumsum([0] + [k for _, k in OBS_FIELDS[:-1]]).tolist())],
        "action_low": u.single_action_space.low.tolist(), "action_high": u.single_action_space.high.tolist(),
        "goal_fields_to_hide_from_transformer": ["extra/goal_pos (indices 21:24)"],
        "workspace": {"quad_origin": pv.QUAD_ORIGIN, "goal_half": pv.GOAL_HALF, "axis_gap": pv.AXIS_GAP,
                      "tee_box": pv.TEE_BOX, "tip_box": pv.TIP_BOX, "start_goal_dist": pv.START_GOAL_DIST,
                      "tip_tee_min": pv.TIP_TEE_MIN, "goal_z_rot": float(u.goal_z_rot)},
        "preflight": checks,
    }
    if out:
        Path(out).mkdir(parents=True, exist_ok=True)
        (Path(out) / "observation_contract.json").write_text(json.dumps(contract, indent=2))
    env.close()
    for k, v in checks.items():
        print(f"{'PASS' if v else 'FAIL'}  {k}")
    if not all(checks.values()):
        raise SystemExit("Preflight failed")
    return contract


def main(argv=None):
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest="cmd", required=True)
    a = sub.add_parser("preflight")
    a.add_argument("--out", required=True)
    a.add_argument("--num-envs", type=int, default=8)
    a.add_argument("--sim-backend", default="physx_cuda")
    e = sub.add_parser("eval")
    e.add_argument("--run-dir", help="ppo.py run folder with ckpt_*.pt / final_ckpt.pt")
    e.add_argument("--ckpt", nargs="*", default=[])
    e.add_argument("--last", type=int, default=0, help="only the last N checkpoints of --run-dir")
    e.add_argument("--episodes", type=int, default=100, help="per quadrant")
    e.add_argument("--val-seed", type=int, default=1001)
    e.add_argument("--test-seed", type=int, default=None)
    e.add_argument("--bar", type=float, default=0.80)
    e.add_argument("--out", required=True)
    e.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    e.add_argument("--sim-backend", default="physx_cuda")
    args = p.parse_args(argv)
    if args.cmd == "preflight":
        preflight(args.num_envs, args.sim_backend, args.out)
    else:
        cmd_eval(args)


if __name__ == "__main__":
    main()


In [ ]:
# Jimmy's binary files (base64), from jimmy-state-rl at c87d8ae
JIMMY_FILES = {
    "assets/tip_start_qpos.npz": ("...
"""),
    "jimmy_ppo_vargoal_seed1_50M.pt": ("...
"""),
}
for name, (sha, text) in JIMMY_FILES.items():
    data = base64.b64decode("".join(text.split()))
    assert hashlib.sha256(data).hexdigest() == sha, f"{name}: SHA-256 mismatch"
    (WORK_ROOT / name).parent.mkdir(parents=True, exist_ok=True)
    (WORK_ROOT / name).write_bytes(data)
    print(f"{name}: {len(data):,} bytes, sha256 {sha[:12]}... OK")
del JIMMY_FILES

In [ ]:
BC = WORK_ROOT / "pusht_state_bc.py"
for name in ("pusht_state_bc.py", "pusht_vargoal.py", "pusht_state_ppo.py"):
    shutil.copy2(WORK_ROOT / name, OUTPUT_ROOT / name)
TEACHERS = OUTPUT_ROOT / "teachers"
TEACHERS.mkdir(exist_ok=True)
EXPERT = TEACHERS / "ppo_expert.pt"
shutil.copy2(WORK_ROOT / "jimmy_ppo_vargoal_seed1_50M.pt", EXPERT)
# Jimmy's recorded results for this teacher (Kaggle T4), success-once / success-at-end per quadrant
JIMMY_RECORDED = {"val": {"Q1": [0.93, 0.83], "Q2": [0.93, 0.9], "Q3": [0.91, 0.86], "Q4": [0.96, 0.94]}, "test": {"Q1": [0.94, 0.86], "Q2": [0.97, 0.97], "Q3": [0.92, 0.87], "Q4": [0.9, 0.89]}}
run([TRAIN_PY, BC, "--help"], show=r"eval-quadrants")

## 4. Check the environment

- `audit` records the observation contract.
- Jimmy's `preflight`: observation layout, success at each environment's own goal and not at another's, partial resets, quadrant restriction.
- Our `check-env`: a stand-in T on every goal must succeed, and shifted 5 cm must fail; reports goal ranges and quadrant counts.

In [ ]:
run([TRAIN_PY, BC, "audit", "--env-id", ENV_ID, "--out", OUTPUT_ROOT])
contract = json.loads((OUTPUT_ROOT / "observation_contract.json").read_text())
assert contract["control_mode"] == "pd_joint_delta_pos" and contract["state_dim"] == 31, contract
run([TRAIN_PY, WORK_ROOT / "pusht_state_ppo.py", "preflight", "--out", OUTPUT_ROOT / "jimmy_preflight", "--num-envs", 8])
run([TRAIN_PY, BC, "check-env", "--env-id", ENV_ID, "--num-envs", 256, "--out", OUTPUT_ROOT / "env_check.json"])

## 5. Re-evaluate Jimmy's PPO teacher (reproduction check and gate)

Same protocol as Jimmy's selection: 100 episodes per quadrant, validation base seed 1001 and test base seed 5001. Numbers can differ a
little from his (different GPU: L4 here, T4 on Kaggle). Stops before collecting data if the teacher misses the 80% bar in any quadrant.

In [ ]:
def show(name, r):
    print(f"{name:<14} worst {r['worst_success_once']:.0%}  mean once {r['mean_success_once']:.0%}  "
          f"mean end {r['mean_success_at_end']:.0%}   "
          + "  ".join(f"Q{q} {r[f'Q{q}']['success_once']:.0%}/{r[f'Q{q}']['success_at_end']:.0%}" for q in (1, 2, 3, 4)))


expert_evals = {}
for split, seed in (("val", QUAD_VAL_SEED), ("test", QUAD_TEST_SEED)):
    out = TEACHERS / f"ppo_expert_{split}_quadrants.json"
    run([TRAIN_PY, BC, "eval-quadrants", "--env-id", ENV_ID, "--checkpoint", EXPERT, "--ppo", "--episodes", QUAD_EPISODES,
         "--seed", seed, "--bar", FREEZE_BAR, "--out", out], show=r"^$")
    expert_evals[split] = json.loads(out.read_text())
    show(f"PPO {split}", expert_evals[split])
    print(" " * 14, "Jimmy recorded:", "  ".join(f"{q} {o:.0%}/{e:.0%}" for q, (o, e) in JIMMY_RECORDED[split].items()))
if not expert_evals["test"]["passes_bar"]:
    raise RuntimeError("Jimmy's teacher misses the 80% bar here; compare with his numbers before distilling BC.")

## 6. Collect expert data

16,384 expert episodes (about 1.6M transitions) with action noise cycling 0, 0.1, 0.2, 0.3 per batch, plus 1,024
noise-free episodes on separate seeds for validation action error. Goals cover all four quadrants. Labels are always
the expert's noise-free action. Each episode's goal position and quadrant are saved in the `.json` sidecar.

In [ ]:
DATA_DIR = OUTPUT_ROOT / "data"
EXPERT_DATA, VAL_DATA = DATA_DIR / "expert_rollouts.npz", DATA_DIR / "expert_val.npz"
run([TRAIN_PY, BC, "collect", "--env-id", ENV_ID, "--expert", EXPERT, "--driver", "expert",
     "--episodes", EXPERT_EPISODES, "--num-envs", NUM_ENVS, "--noise-std", *EXPERT_NOISE,
     "--seed-base", EXPERT_SEED_BASE, "--out", EXPERT_DATA], show=r"^batch|transitions|driver_success")
run([TRAIN_PY, BC, "collect", "--env-id", ENV_ID, "--expert", EXPERT, "--driver", "expert",
     "--episodes", VAL_EPISODES, "--num-envs", NUM_ENVS, "--noise-std", 0.0,
     "--seed-base", VAL_SEED_BASE, "--out", VAL_DATA], show=r"^batch|transitions|driver_success")

## 7. Smoke test

Short BC run to catch problems before the main training.

In [ ]:
run([TRAIN_PY, BC, "train", "--env-id", ENV_ID, "--train-data", EXPERT_DATA, "--val-data", VAL_DATA,
     "--out", OUTPUT_ROOT / "smoke", "--seed", SEED, "--iters", 2000, "--log-every", 500,
     "--eval-every", 1000, "--eval-episodes", 32, "--val-seed", VAL_SEED])

## 8. Train the BC teacher (no DAgger)

One training run on the expert rollouts. Logs validation action error every 1,000 iterations and closed-loop validation
success every `EVAL_EVERY` (200 episodes, all quadrants mixed). `best.pt` has the highest validation success-at-end
(ties broken by success-once).

In [ ]:
RUN_DIR = OUTPUT_ROOT / "runs" / TEACHER_NAME
config = dict(teacher=TEACHER_NAME, env_id=ENV_ID, seed=SEED, iters=ITERS, batch_size=BATCH_SIZE, lr=LR,
              eval_every=EVAL_EVERY, eval_episodes=EVAL_EPISODES, val_seed=VAL_SEED,
              quad_val_seed=QUAD_VAL_SEED, quad_test_seed=QUAD_TEST_SEED, quad_episodes=QUAD_EPISODES,
              expert="Jimmy's ppo_vargoal_seed1_50M.pt", jimmy_commit=JIMMY_COMMIT,
              expert_episodes=EXPERT_EPISODES, expert_noise=EXPERT_NOISE, dagger=False,
              maniskill_commit=MANISKILL_COMMIT, architecture="MLP 2x256 ReLU, tanh output")
RUN_DIR.mkdir(parents=True, exist_ok=False)
(RUN_DIR / "run_config.json").write_text(json.dumps(config, indent=2))
run([TRAIN_PY, BC, "train", "--env-id", ENV_ID, "--train-data", EXPERT_DATA, "--val-data", VAL_DATA, "--out", RUN_DIR,
     "--seed", SEED, "--iters", ITERS, "--batch-size", BATCH_SIZE, "--lr", LR, "--eval-every", EVAL_EVERY,
     "--eval-episodes", EVAL_EPISODES, "--val-seed", VAL_SEED], show=r"iter +\d*[05]000 |Training on|Done")
shutil.copy2(RUN_DIR / "best.pt", TEACHERS / "bc_teacher.pt")

## 9. Training curves

In [ ]:
import csv
import matplotlib.pyplot as plt

rows = list(csv.DictReader((RUN_DIR / "train_log.csv").open()))
it = [int(r["iter"]) for r in rows]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(it, [float(r["train_mse"]) for r in rows], label="train")
ax1.plot(it, [float(r["val_mse"]) for r in rows], label="validation")
ax1.set(xlabel="iteration", ylabel="action MSE", yscale="log", title="Action error")
ax1.legend()
ev = [r for r in rows if r["success_at_end"]]
ax2.plot([int(r["iter"]) for r in ev], [float(r["success_once"]) for r in ev], "o-", label="success once")
ax2.plot([int(r["iter"]) for r in ev], [float(r["success_at_end"]) for r in ev], "s-", label="success at end")
ax2.set(xlabel="iteration", ylabel="rate", ylim=(0, 1.02), title="Closed-loop validation")
ax2.legend()
fig.tight_layout()
fig.savefig(RUN_DIR / "training_curves.png", dpi=120)
plt.show()

## 10. Per-quadrant validation and final test

BC teacher with the same protocol as the PPO teacher in step 5: validation base seed 1001, then test base seed 5001 (used once).
Also saves a few BC videos.

In [ ]:
bc_evals = {}
for split, seed in (("val", QUAD_VAL_SEED), ("test", QUAD_TEST_SEED)):
    out = TEACHERS / f"bc_teacher_{split}_quadrants.json"
    run([TRAIN_PY, BC, "eval-quadrants", "--env-id", ENV_ID, "--checkpoint", TEACHERS / "bc_teacher.pt",
         "--episodes", QUAD_EPISODES, "--seed", seed, "--bar", FREEZE_BAR, "--out", out], show=r"^$")
    bc_evals[split] = json.loads(out.read_text())
print("Test (100 episodes per quadrant), success-once / success-at-end:")
show("PPO (Jimmy)", expert_evals["test"])
show("BC (distilled)", bc_evals["test"])
print("BC passes the 80% bar in every quadrant:", "PASS" if bc_evals["test"]["passes_bar"] else "FAIL")
run([TRAIN_PY, BC, "eval", "--env-id", ENV_ID, "--checkpoint", TEACHERS / "bc_teacher.pt", "--episodes", 4,
     "--seed", 9999, "--video-dir", RUN_DIR / "videos"], show=r"^$")

summary = {"env_id": ENV_ID, "jimmy_commit": JIMMY_COMMIT, "expert": "Jimmy's ppo_vargoal_seed1_50M.pt",
           "jimmy_recorded": JIMMY_RECORDED, "ppo_expert": expert_evals, "bc_teacher": bc_evals,
           "teachers": sorted(p.name for p in TEACHERS.glob("*.pt"))}
(TEACHERS / "summary.json").write_text(json.dumps(summary, indent=2))
print("Teachers saved in", TEACHERS)

## 11. Watch a rollout

In [ ]:
from IPython.display import Video, display
videos = sorted((RUN_DIR / "videos").glob("*.mp4"))
print(len(videos), "videos")
if videos:
    display(Video(str(videos[0]), embed=True, width=480))